# Version 4-A — clean Bengali Wikipedia retrieval reproduction

This notebook independently reproduces the public Version 4 lexical/Wikipedia method using only
the official competition input and pinned `abyaadrafid/bnwiki` version 1. It reports the original
same-OOF cutoff-tuning estimate separately from nested duplicate-aware grouped validation.
No Kaggle submission is made by this notebook.

## Input and safety contract

Attach the official competition input and `abyaadrafid/bnwiki`. Keep internet off. The notebook
authenticates official hashes and the complete Wikipedia path-and-content manifest before parsing.
It never prints test text, IDs, probabilities, retrieved passages, or individual predictions.

In [ ]:
from __future__ import annotations

import base64
import hashlib
import io
import os
import sys
import zipfile
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Materialize the synchronized embedded runtime

In [ ]:
RUNTIME_ARCHIVE_SHA256 = "f2c3cea0b53659573be2eb869218015c26fab4d1b6b160c572ffd4bf96b88649"
RUNTIME_FILE_MANIFEST = {
    "olikbochon/__init__.py": "a77645398a46dfaa17cdc02963e9dc72c82f264a04c15fe5de11a0d94cea5cc6",
    "olikbochon/data_loading.py": "541e889021cee70c984152ed203ce95c0e835de58b67015798a25c3241a519da",
    "olikbochon/metrics.py": "30a8268b971a5c9d737660d0529f4983ef3f08e0d225fa06155ff2d6fd9e4175",
    "olikbochon/modeling.py": "0666af66702c0a2cb7438dae3925e79c1c897798271744ec4072035c2bc552b4",
    "olikbochon/submission.py": "fff5c7e2df7b305c87e043cbea123ba578dcc400836450ad18ca03edebf998af",
    "olikbochon/v4_kaggle.py": "a236f4d5c614d6f77f0f7afdef41ac206175f0cae3400ab5458325724445a5ca",
    "olikbochon/v4_wikipedia.py": "4a3a5632e24eeb15dfd7ae2ca643583070b0a7767adb6068074c63baabcf224b"
}
RUNTIME_ARCHIVE_B85 = (
    "P)h>@6aWAK2mk;8Apodx6~|Hl003D4000&M003`nX=`F{V`y(~FJE72ZfSI1UoLQY9m=^4fG`XIz&=x0?uo(xG%UafRtOGQvW!F="
    "=I80II7D1(RWXK{ZO<xjWYu28vdd}1#?Ivvthw<YyNikow}^-"
    "#5Z!#WcM!n@nBo`{h6hkf0|XQR000O8001EXy15*S!2|#R)C&Lr8UO$QZ)|C6VsB$;Z*DJSVRT_%Y;R#?X>MmOaCx0pZExE)5dQ9"
    "8!Sa(dI6~~$ZWh1;($+>hI4@YXei;UVlBk%QL=q&G#>n#DcSlOH<zz!T!>}dtc+c_fxrd~b-Z>+3M=F-"
    "_iWesZbBt7q6++$<w`Gn<{=f|>gv}K^SfUIuoE4Vjs@B{ICzK?PTL(XnE}@i;Mx{|TNz<}%jp1oZM6H!^gh{C!!v%IUYR9(BZYz;"
    ">uYX!4!*b0WPIPN;;k92Rn_BM(vqb7{NHbZ$0DN^Z8jUuWA6A>yr*!>gv%0>#T9GB8xHrokv8>h=w+Uj=-"
    "tyzk=C`ZsJ1~rBt|V@TjtSKei*{)y@x^efA$Z=t`lQi#bhBQ6`S|5>lipok-ELOt-RH~ce1VYVWIj9P?@y=i7Zb*2^T~8JXIZw$="
    "0%>L{dhJj&&#u7vdHHdW9R4dNj94lS-"
    "w~lY&y%P=a9=$UNLLQCkXy4t3=`Rw=zbV2;Lj+ALEM=fe$4A7bYw($hR$*gmq;lLh8nm2caqkrGmn8@)e4SrJE36bKJ6Fd0W6y!A"
    "oMd47QP$qT&%&<H8pqCMQ1;>r9XM87c@+Xol&pxnthrH{rGlPz#<bPbcMRq${?=H{q-"
    "Y<<eOXitY+=k4##2v`9h>QS4{Yqvu7UG?x)I8I2Kg3$?F!2natVAZ3-"
    "x|AKu34mVN7>a1WFt!07)jwYwm8To<WUpyumrSaawK9;1>up1tEe<2LRf!4B{-}1-s1VM(pJ%lpxw9S568dmej|6Tw+6RhPL-"
    "+V3)2l}M8rG|{k0@4I*$C|0K6uDqkUss!NdA(1yHbOc~E`Mw}-X&I)#5FJyqDXR8HMO)cdAB4-"
    "4X{swK`OmUyg{Byx_Z<+cN_!tZpl+6ARkAL<L5C{rh)y0>0u<#w09bJ5eEwn{kMBdRCLp3rEB!M0WM{nNF}`!I`sT|+rWP$!b%p!"
    "36r}hemjy40HA3IQVVM#^?>0WRKa3-"
    "4CR#Hd>;v7_J{&JSjS|}qX+EmF^MvzsxjzXf)8mN_qM=sM1ExnaKNvCe}2!+a19Iw!#FIL<mq%wCeMTB@dh{-"
    "cH!r~KQS4)@F7^}8S4)ksE$+Q)YE#huLo2ss+kU<2hiA)GX(c0=no3i!LKD=GC}8BoIV^4VSwJe;K5$xhg`sfwI+ws|33(G>k)i7"
    "ou`oH|0GeTBt^%g1f?J<IM>Tx4b*UG)w2xDUk)bBx}D}xV8&uL5VPLA)iAsQ^!kMd<-rLs26+h$p9Xop!1I9oO|GHO02=@PUD-"
    "C^nV=y!sX)zKwIjCC@C@60A#gw6-"
    "CX;lrB^bH13)kkA5&jTe=taZIuy0;{C`W44ulL5y{o5VNAJvD#(G2;+;a*`>1TuuXXgustYP$8xixwT(1VQ}Wfxl9`^2>FJ*muc^"
    "8iM8QbInC59@u?gU6vUr02MR13`bI?t8JIH^&w8_W+LrWUiUZfi)c*c{&1yT+&yZcJY#@LFCUv+qOww+zQ}o#$~v3ENSD)lk?ZF3"
    "7YMY=8!xjc!JMSuo=|yKw;YzAW@*<-yw({k$tP<rH2W;ojIjXD!5)(-XD-3x-"
    "nO#u=Lg5Ji1RDc2^e}X#GD>O9KQH000080000X0Kh|eMg0o^06ZxG02KfL0B>w*YhrI>Xm4&WZDn+FX=8IPaCyC2dvn{k5&vJG0_"
    "DujkSVQ`(x%h8)lEH#oy#<FZfxJ9qtS365|nu6Lj_33itB#%?gAhPkd$KIU7N`y61iCH^S6tIJkR^UYM#X+B=wdvv90oi2$r+B&Z"
    ";sbv`omhDw(JW{=*WB?2hJHLXmFjdER7_@~R+Fls0w4Srn11*j2m+7G+gSGNEbWD%Tr`DP6}c;WhhL!^)T`BH&EpoC?82OJE);ih"
    "94x%1!(7tlUo~`g7S7yFH<Tl)LuP4m5!W@OPJ}<>EGHl$X;&8Ni4e#|@|PJ`!=oSqQ?GX(K?^sGvZ2ACh#g-u2j23Ckm6I-"
    "^w0LH9Ic$v-Z@Xxai*p53mi7%Z#SS$bTZUB3DFA-cXfyEzZZy2<h+k~AU7=^D1!@oG)iS)SFP@Z{#(tMlt`KfZr+O%|Z;lyX{bnE"
    "!GXl0VNwl7Nxe0-k4O{ii<$<ZCjY&8D--#k-%*-$b)lGg#%N**#0}i#Pdj_WJ5$bTN-"
    "Ue!Rd}1&w(XfyPypdXv}h&#td$`gL^v(|6}r?>?Mg-kiNhTIrgZCXU$sj`6HuWleK$^5*>F?ECjOkr5XZyaEm1{u5T(n@$E4@6J&"
    "*e?s7o5j&J$?S<;eVF|-"
    "Y0OtC;i6I{x@tKv28{V*BA|DZ0bHzlH*Qb&hMD^Kb!`X(`j8sj%YidC_t7%q3B4i~gSW)pkNh?kSO&L+41SIH5@TtlRWYsw>V<M7"
    "{oS&k<ASyYhdricWOy$MkU0Purs8Bs6b+gOa5{Z|h<{<|EDkRhCboE6YFJ&6P{-"
    "LMn_hnr*@fP#~MHxTA{cM4VjN1VQn0U8UJ*<TyhhQ@tj529n(OT>9OobDc+Pg+5s<yH382;2?jM{Exq24Kxj{<V?jU{y#2)kXr2W"
    "kd@Q0!Qp$t0JX5Z>gB)D`*8m){rjKw4wycga%csgPe>`K)}E!a&AKE)a7KFNr_K?VE)X+&7v80dPk?zR7!SBpCq!MzMtB*1G;asv"
    "$ru<G>;?%|1e-_8v?7xPCA;dQJ1V$uU`I4pxC>889H!<WLK>pR#&O03h-"
    "#&T4JM1ZcZRLXIF5kU7>;k`7?ajyu?}w*+5|FVS{=kYA({0A5m@nWy8yZ_(bD;)LBc@UhaVya2TQ%m65p>@LGvv6uxxIOIBi9@qJ"
    "Ct}Qk{g(<yboNm}cE%J^tZcCtH?AF8qduMUKUILT;qY?R}!eX%mL(W$we!*42w1#3SYG3d1s6*LAdsNeP&JM-"
    "!IPUgPK*sbgnZ5zsyx_E8CZf-w$7wKfG4b;%zIDKF!?Q%jGC-!T3s}f`_30#McM#LMW|)YEH-JH?js;~(h80BZvPC{_Z31PctMRA"
    "C&TT3!4kfjr#6NA`JP9TB!*GZzhG+K)RhrV>j+KeK<%9E0+S70{fz|O2hvarHYOR*m!_|=`^RXrK(Ir+EnHA>Q*a9m%%mVYQx8Sj"
    "7F%39&todlc7NAvq?5d+LgZtE35E6ksCEM&z@e@SA;pR$4wluW~>1(+Kp;m}VIK~K17?l13qFB_>##B6k4h{ku1Co*w_D;;+T}-"
    "zyzaS5mheO+cp=D-ek}M@~Dlm(unGnrduv#fRg%d(klp-LQ3i1gncJ;mse@-"
    "*O$PbX6?40w8`>A(h^pPt_(ZHc)&0wEWE|zB*fFev&@*to?V~O7i5IlN8D_E-"
    "{SuDDS)7o13a5jB4w!m&lyER%L%<;|=eG?^Mb_%G3pF2Y;On@VTyU*c9cYN1zXMAy2!H(G63H20O$lJ2|RNCEPEAT`6>396`uUPq"
    "I7f8S}q)N%dSkM3H^>c@ow=&FBKp^OCO1~9<p(1B#>p)J>OHfVL!iS~^T^=EzGYcKHjk>g@A_gpjuwN`MDCi#&{ojp5Q2f_Q^4qL"
    "Tpr|swVaVQ7Z=7UTNT;NqW>7?yvy~&JN&Xtos63I!Ckv^Sur<4{?FJSsDtolive)lw!D<+<5)OukoVToG=pG{hd8PV)JOnzWfj}7"
    "tK(MNzJS`9btz6wTZYb(lvnyY;tys1G8KNNgC9Ph32eiGDwO>Y33bA+v2)S+2G-"
    "nItGq8OEwGOna*}^_ww~eE%p%g*_hM%)iwJO1(##|m!MyonD#=54PqAsZ+VAy>~fHB}u`%R8g)RzD&J}a9-"
    "y2Ov~rcy26GHOU#sX!%O!+FE%3vg5b`&xqoKXBhpQ<vS+5@n@%JI}iHQiHy|24e|d&nxUbEDcw>-2=G|V~3|h3>yr0?F-"
    "Z$Qx^w&oFNx0PkJgor>jYIL(@6;*LFTHUL5G@lxtj5_AL+`?m+s-^CQMOZYPaw21)wl7S~W6-"
    "F;)$poKd%OXzx?CqkZlGY)%d#PH<iC{%cOU?5^ze=Jg_kjYjlD*-PhIM$8q__4ES+yRR8OV|0^)f*chY(bpq)sce!c%%e-"
    "m;H$hQ*z^ULN_-"
    "8<U?ss+UbiCMfW`abG()oiGEwuRy^HkK91a$K=$}a0=GJZ*=Mll0K1Y!9;sV+SC1r$hWfqC+aA0d&$cEyjP|Ot=wd-eVbpp;L4&p"
    "Hs!xa<k<bF=Bjlsba*)M%?hh{FI9_z#9P}vp{m_IXXH6y9(Q<7WWwdBpFwczzrH|AX=_7PeYX$8O%Z8zCdCMQ2gvV=ek4{!&i~Z7"
    ")YUS+Zycy%?7Ew!~)o7I-"
    ")E!HzkPJmqtW5ECmRpuJD6<$IXPq_u5ssy5IHpn1oqQMVhmk1ha=WeItwcdD+m8Fmq8W5dcH)F1xz}+~#rn%FleJqolA}G_5BWc("
    "sE^1Jl0(L^m^d;3|G}QWx_RnnKLEn~3vE;@_p{q4*PsJkzSKYI-pB14{DIBdzNd@E;BYGh;@xF5Z`W7+_-"
    "MX2gYVTQ)4^ORNr&TiInT-"
    "S6SQ|IKS7V9?s^OQbHyPZi4jHo(s_KAsufd_eP%eanBg=Bn;)cub7VUPHbdVW((hQWb0Y;%qhd4Oq1}LIot2dDAu^P$baxY1(3<H"
    "&0Tl9UyLFM3P&GF7cJWdXxB9D;fkmK@FAE5*VIQ=Z-"
    "!fci*3*LC`#}K5dL1I@Qf=!9Y^`pEPLb8Y+Y>9Y+${0TkX(rbkuU#uqx5*lhp2kmfO$KIXeb%D&I<oWC*ZA15-m~$4vVbetkiX8-"
    "v!u&^B&>cy~@yLmumDVU7z&s0lJin@Mi^dCs_1}a%{_eeN-"
    "5^#R|!bP!2TZDFJMe6<KZ1O8zoahXXuA{F_`tgqgJZ5xA32TUN@GL3it<u2%)hUFUnk?qkLhp|1?fG}|;dB^6uBnRPw*>^rM0?N%"
    "L5E$eFK$?5f-5#~2ZF-wF&nn4=~jLN#7W|VI*nH0aEM=MQ7+Urh5vvfJF$k#pV7^C&r{B5C~-"
    "cLXDwH^0e2}`{n&#o@tUA{fFqK>ptCsFkQW9m_-D0y&&d&D0GKs8`!E6-"
    "y4;L3J7oqzR+b5Xmh`$fg^8VZhb4@&Uq?5oFMO3uhO+iX?zoroQa5d<vu-"
    "o+eHZI6D8X(Pb9>%ERuBRcd_TgDZ4AOWZVkP<P^P%L#PtLK_3>Ga(^g(E<n`0h=;$+KQrI#{&y{a1HE(G&&c7}z4FIaMcV=jvEqw"
    ";jA>Ll<By`tHoTy9MeEz|GywfjeOZ6FpQ$<;^Sy0vEsgOFinXJ}q$(sEjcIU-"
    "3X;TWJ;ax(pBOXtP7JKAjs_GpSPd61VGHa1}DZWzsedeu`$j;Z?J9?+u^xi-3@9sfh4m1-e%2-"
    "MXss)#S;&iv01D{1wfxVOitQgcs1NeP-"
    "$o2etxFZL)&>AAc^Ok~e1V3QPNT8LHk!lN>4?wg6t6Jx**t0bEjXi{A2mPcx1*Zg`;)0@}&$HC^ks|1qyEsMl~VrEba${G(f8+{i"
    "ksKy01%<U{e?NR_zj$!^BS^;6Ll{ve*$RlSY$&-"
    "bupfup&SBEWL>vPbRw7$MqKc_yz{XJg?EP%gKn<=*d|zCr$JvrFXI{MZRVs%`!JK-"
    "<AGg!Gz9JFo!owV^@1o%|P2O9KQH000080000X0H*h+&UOO;05}N%02TlM0B>w*YhrI>Xm4&WZEs{{Y-"
    "w(1E^v9}R!wi?HW0n*SFk!sSVeH0_F_OS3@2^_B%6ifqy-"
    "#<prw&bgd$avTE|`NfA3I|B|EW!qCJ)eo8o+MX5Q$HeBZxQ=__Q!luR2SlP^!CP#Fl4Dw*?jrJ3bQl3Z!>4BEK;W$gRD=jB=zgwn"
    "jSl?F-"
    ")FG{5?VNxnv8<|!oRYE{opTumH?iC;5&kAG;p4T48swj7a86wNZ(7Xy@TE;oB`VM%rn$?bCi~GH!dCs%^8E;UU{{{NkR&a^`w8^P"
    "8xKnFxEKg^!)?i#t$2KJ|!P)Kgr_KFO5;Y_8rImV~HyeMxez>QP^YQ$KB;?1T_xr=_n%<B9pwpZ2{AqUch{wYL9SjEE<hSwcz38e"
    "D-97wz`#8Uy&<UPh#shSnK~7c`7g-"
    "Z}5R!|ZyQ@bY!G$PHG=>JmZh*iWmWEW)RubXdDxG~wH_X)wG?`B?Zm*|Aw26~_hBcDTS6gC(_a9~()Izrw-OjDx=&26I-"
    "RMUj&fkIa(a<d>nb@H;Ip@VKxz?<pn#naJ!DU2-;c2~sz0rL-$p<{n*l<a+Jh?pY(MjXMX}PIZZJXIVnd=Hps|mFQoiYQ-lnHZ2$"
    "ym)x%F-01HOXfs&*)`#S3)AoI2Vf9tKn%at6o9Ul38p5nP@-wr3znGFWXNqt8-"
    "OWx=4I6JRyEp?{|+UTB7T}|K1nQ72Tc<`mi;wBJ$HeX^ju><NvEQ{uR*6dKcEesbI*RIY)X)PEcx(=p^a+t#@UV`09`qysgkdAg-"
    "z$`z9G8JNC!*jdTlqy|IbEVuIoLIHU{gfJ_zC;QWE)kes7~xHTaDvg6WV5%~L_@S~0h!-"
    "xcapCU^7hj1CT3@z9S1TD2%u@x8G?t+@FN1gYGxJhM3-"
    "LzXEip!`m91$*UP3#Dk%$U|}*HQbd3A6Xjk)7QbW)iu+6g=g&aYMG;Zd8S%*O&_1b=`z|fpfPiBtb?6-"
    "4Xo}YIU%&WCdhcG$_sDFy&=$P!9$Sl9@Wlpx269Ic<ttFq>}n+PPp>Dv2+!3OYAQ6Lh1_v4kLl)+!`F;P@LIz{&BG<e7;IZnRc9$"
    "o>1~+oW~&t=+^{32eV0{7y23oo(IMi)CE41^QP1eQ#!@*b#+5_($E)RHD?7@zgee1~qcwViY-;_AxF7%W&!a4Nyx11QY-"
    "O00;m803iUj51{@h1pol!5dZ)f0001QY-wv^Z)0e0ZZC6nVr^-"
    "2b7^mGE^v9hS5a%@I1qmKuh99)246_w?j8a?=(Y(2ns#Zr%R(tewv|SlE&C)nO|IAfej~}U<s{xs+vQAYENQIyH1mBL$@l%G<TRB"
    "?)JZ{wpd~|T$%N!}Dpdos3~g{uGpw2;-}gOl#mfT4@v4?JCvl8uQI%YRDJ!Km6OBz;<~h;xh!IXwYq-EwMcGq?mgMi6u#|XSGs5b"
    "k+950ut87;V2%rQ0s;rr|%;{fAnXUm(K@@ypXAV3B&*k;SaygD?XYuXrES`S*JiVJQrZ@K&*O4Bd&A&~r;_>fe(_CC!-"
    "rY9Vkr#TNcRyX;$CtO)Ulun@bdCa_W`2Zx2#h3Ksa`9<QOP)As=<k<V2MOELvQ)z(_+3{&TnsyOp)V+<b9*r{CawGv6w2emjgf(<"
    "aenVrDEgnt)XWSujrO!s@pLg4oCmnG{-`WW3sIXrv+gW=glmy5EDI$maX~MmK0OYOCCVGCi*p;XtzL>J4^-"
    "1(DRxwB#Dp+(uLt8j7pBM%ef6La||X(t_k8m3(Da`6>yGIX=vwVWM#QXVl`j9;<zAz{yzbJjILnrnQDdT*FVus$w-"
    "T(!AlGfLvWxkdZuz+))Hk@R5{)OFBLdJ3d|%;ah~rGWuj6vh9Q-IjYiE%S4h@X-T)W@+v|df5dCtFTm~5k>ftIL--%&ssGgNS-"
    "72>b2MxKI9=+sL$iQ_FzD8(O)}T}~)`#=zx6_6ewzhasoO~Qn!Eg|cFxv&;=}`T9g=u~zP*KARC?6#RX=C=}hfTI(3W|9mJWeRXd"
    ">3o_i@^Noj4gp{MCf!SWtmSbLNy@_<BE>tt|9_pwv%P5r<r7=8!m7HX=P}Jb3r_(L(Oi#p#U;?7)R*r5f+~3K^X4StK+80O4sfa%"
    "}7j>k!J3a%z(fJ5F{S@8vUzD3d&bMu=xzrmTW-?mIM(*9yn@t&j|7I8AfU*BfVcYaY82Os9E#@Yu$atq3+@S^=<+$-"
    "J)XHtJ#t?%K>c}#vb}SO52<;2en44x@`2{LSF}38$AQAz%pGM$+aVX3!!uWG5I+DhedS6rzs#m!>ysk>7Mm!WvTO}LvO?0c7AaA7"
    "1U9}H_Z+W1TT3;cn3y-"
    "9N18fMCd0(1tMalv{BVfps}hAXKH)10ovC$qCx}hMBLZhzWvG4J8dC)dbhCZg{<+$ZXh&^^b)NgoA9~<+lTl)oxKy)u;kR~ppWSQ"
    "&)cZ@PK1_zC@YD{r~?ux+m^Vbrg?TyE(Qw%kUXQRsK>Sl+C%UIaP9H^Rl_xCtH&n=4o+IOtYz*JpVt)D+^EhRZkm$bY|d@TFi>O6"
    "Io^R+R*kUGjbMK{J+XovxT+JkUYkiw)sD1<QP^AL`q?_|sa2C4vsSX&h2N_kE`<NTdN`p!@1?6Nn=gvmla9_RH}J+XvJEm`R_FJ;"
    "CSiMznDjCONzs8eLH*;m@YU<8=dpuL_Zt1!<>9vAp02v?0a?Tw%qbkH#cBFbSH?Km#c)ms&`-OGy)-"
    "^RPt@1U2SlS>nHgQv$9?_iGgU?^0{@sgwQSl`*@nOgi@`@3`bxM??pg0*W0vinz97EAz%n%{)%dad;Ts4#d=6_oLA(!-"
    "VgjzvBs=c=;5cUUIT;F~y3QH~vXU17@E_PEAm!-%M`Y6<t<){72@6CY9YfUxq^GUpTe5xnXrkWOKw2K`%VXdCfO64ZkWNwF9-"
    "bkgxiRnXz-k@oIfAsN-hWU_0|XQR000O8001EXuG{zV-wFT#Q6&HX7XSbNZ)|C6VsB$;Z*DJkG+%3BXJ>3>E^v9RS#59Ix)uKJU%"
    "_c#WXw@}i?q1N;0t(-vl=%|ip0&`mIZ_?Q4VipN|%z7xJCZ^&LJs@5+!FHRupzjo)_}G@jRsGdDkpWj-pJIAr~wrzlPgw%19=ods"
    "dV(EO{nKl1WnDGZLjN6y%1<0!NRJ7DTdKW^olES>SozWRl2iM<`9IvXYEa!gqNlOA-"
    "o^l}cJMnV7NPi%isi=V@4inw{p)!}31mo0|Cx{L~XA-"
    "?4f$XELEtR*8~HGrP~Z*w*t4v2Rp~YM1XxSP+pn8bcAo0Q}G6daQVW9ZF$#5$sqgc~mqCC5yS*DyU7<FfSN=0&>u47>a7M<At)0;"
    "n`F?jcGeRBadOq<FI7bpy7TzrceBV=b$C5`TYrje`m|Z`s>w_e!M*Y`uXB|rUoysR)1ezEY9ia^2_<<KbNQUYPr5zEpK3G{po6X{"
    "ps@jRFhsUE>15$)9dwOy)+{(sS$g9vHJ35xuzdKEv^<H*UKw2{@d!;6)fOly}Ve{&x?!I+46c#uRkr`z5ifQoG#XjYhYiU4h){JJ"
    "}xh=m-"
    "O>;y}+>nlB?y}z~~JuV|969&77^y(H>tf*4E(X#XqRAmA&fq<r+46z518g{BO0Duh7Blkx4xs^CHUN7X?owJY|I*X5g|4J@$AU>X"
    "C$(Kp?`D?y{JrdN9qx7y@ILvy!VIDItTbsVqr&#DOObH!NkbHC2SWJhdajnrlJ=WXrrVYcjD&A^kSthNqyhUTzU4jMiJ(gV5bBlz"
    "Tnb1YedV7Vn;hvgAn^l|3@t!U)}TGMU6IAynzZT&8Fz=Bv*WOus2P`USsFlu(`!UT{&Ap@<CCjNm4wdLI8t24zE#?@)UD2ezM{xZ"
    "-)qc2iZ7fM7=OC*gvq$pC*9{`6&Bp`UW_3Mnfv_|(;Wdq7`D)!n!Xkp~=#(w0eAK1^t`=|TPkTwg+6aRsQES4H?>z!@r=X@(iuWL"
    "Y|$LSfg-"
    "2Oe^f3FgRGvuaR=ykKO#&)HJSO!|rURiJS?2Y`t#Vx0aRO0ZePG@w+3J2(Sgye3wjB*)5!CN`Wt*7_4w;%BwA6BQP>Y7Y=TcPftB"
    ")q*cJJdT+lbMoEeu{R^|S>RX6p3C}M4RrED57b3m{gl~L;k`_B`ojvtD&}QBgu90NZ{pZ2^FWApX^q@E?1MCWV$z=?SF>A}qxI4a"
    "itf1ywsw?4aZ4K~foO0MeEvzIPx_+t-"
    "c`Ieb1eO#EUNe2*|NKEThpW+FzeLXF)jifFX&Cb5rVlH#BiV&zLTuAgc)I?zy>5NBF^V$VOlWE2QXtfe+PjEXDbCm6!W!&P$hpM-"
    "oJ%6Yqu?;EQb7>dsUen{cL9HFqV~|52{hZBSEi>i@YkS%(7AkGsGr-"
    "`Q95%`9=|VQ!VK!lMf*N5WOKm4i$vml+ZSXj_ay|jtn#LiI?|TRl=dM-"
    "C`GnrJJDTVBffHo^Z(98CHNcGOTP+?HdXOPs<Wo!Fi{{@~6(igWW^SB_yx}@!FDL_h*O<nmuU4PRwx$XDP^Y&R4&++O%^5?}2*w9"
    "P<^U2@#3q)LT^0oWcg95SLvav&1@zq5X^r^oH2s{*8$(9B*wCpj`+1w}m#5;9X-"
    "bDUzl<S%LDs6WgAB{Nt{?lXgqD?(E&H%if;#E^}|L&)ResXdmsgF8>h0xc=zA5)VUShDpW+mC*L5wBX;^m>i4367$le9DX%Uoe%d"
    "?Ix2!@7?2AW+Ml{>su}y;lpRj8?goL_Z4rIVFzu{|8XmQ>hCitP@@u+VJcxFANjJi~o7z1VM(4cKVm3^(0X4%hdeh3n7)x2n4g_i"
    "6cvKyNCnRN}(LL>lm*iXB@G>YfF)i;UEAF#22EKMDL67X#sr+668lw`G)?2b<b;z9v?=4{$Mlz$xPls2~Dl}-"
    "$yBk<lBkSM_?x7EW;<lApm9w!0G_yv3tG3nB7aYSms>+v^T?|O1>WDOdtE3ytl_CmLKr}=_6Ns%ISZTwii8E)Ubxu1s;IH`)Kv1)"
    "|Ka(|{OL*Y`B^$d?OiL`=A_ah+z)3@xIsi1XA_)ueCSgWC-po`PZ?r?d7IBTW3jzrG8xw)jp-"
    "#T}Fdj370{@&EwA6!MHK#LSSY5296y9PZYEoOFrdKM1=5eY*4{&jHZVrOI){iaDbd|T`CM=rY7IOXnxffw&4H0qSac~-"
    "z;h6*+;E$Cfs6G@svuB8UfIguN#g_TqVnkvD7ITfPJ{<R}Q{oh&Ity`zZuVH>dOWK50aII|^~haNW>no5`G=L-"
    "uvoE<6oM|)ZeY4fTw^=@d=;AewsB|OYCUbNpX*-"
    "I?$gK{;=#G%E+mMGN6+ya?jI)nTAJ>gpLG*${5*OuFz$5Bj}t3ZAQcK$YR~(z%yQi^&PELDCdQ!gMi=IfUuC?$sKMOaAF{Q(+}7B"
    "mE2%9%sA$LyrZ7N|DLY}U=rdyl`DISt2JfdqSSWbj1hg)IcIE3|SNg4KW{$R*TReN-"
    "umla)%TW>S?=dU46s={gF=NV}oFOAB<OrKlhbo#7>{AK;2f#^p$NH8;AHW7Y+MYl?(BwiX-BhNv``%vpoT%#`#j9_W44PIW2$#R{"
    "-0v>OTxB7#S^`q}Z)fBu3v`d@4j_^Nt%g-v`igf(-"
    "qtN~_x8FyVutRv{k|g09C*Pq_Z0qAr>=dwVx?z&0Wr5~zxSxGks!A{;J4aU4-jZsVH4y<aN?^_B6W@wB$5n)-"
    "d~BxO6~Cyes=h?N=B^i6G0DR-<amr2D&T?5JVyvc^iEA`Tgr$HPCNXdu;JIt#pl;(9zurZE>j-"
    "?Nh3j#HMPX3J~><Ot46ZbxN`XCWr6O9b#2~2k+<|(sr}sx9|;<F%X!Ri9*>gL^*xt;k*AvN*8brUh%i~voYo14Sy?EH=59^3!b3a"
    "i~I21`wunW!S{?tNFK@peH)5QK%1WS(6I)Ffr%e$u5VoyLC*!r!v*UzyK_Be$$r-`W{7)Waui`yGIf-"
    "@lRfsT2shwkOv1>zUZ`W&$T8D}9mX$KM1wODzOD4Dq8ruF_{G_dV)mnB-"
    "A60S7*4ZFn^J*=WBhr`@<yI!`w<{|@*}TnjmSdq9MA@Y05q}S4~a_|drxb)FwY@ApQ(X5MID^(YgWT!NU>H|JAoX7PlT=I(-"
    "00pyaDWN61sw6PciYxWhQn^xw+@3d=C{8{Z~KQ2706}8-"
    "Bl2R^7Dj?rBX#sgq>*9*a`r15`x>)+(ubrUhE_U^~oUZ>WEH0HuKMbexS}9$5T6=s(Qfh5i<|h2ULtT9Ez+albY$R5-"
    "~W@eq3L^1vZ=Ad%cQyBrtOA?t}?&k*>uEhtcs9uBN%kle*th>=w#e=c8>I7B@h5~fie?42+Shi485=r>NPz;=f$?xvw}Bcg3uF~q"
    "waY=A9S|7NU*81bS4k*xu6Lo5!&Fg%0MV@QCtthNZ<n%6jkq+eNP%ZIN?9e5j<xqc;ame-Rp4y{~)b}UH9E4~950>`k3RTb*IeUP"
    "L2Swu6v*lq109EeXvhAhB$D%}+iy@T4qnwQXn*S##(FF$RQoK+=0WgNKswcKXi`daAg6`$Ex#g@Nb^^?Un>Ld4p<$umi#s5S3*jf"
    "kC1O@o0`&v9&)$5^q+lrs&)!evi@*hx30|XQR000O8001EXlOA4Ya3cT!!hiq(8UO$QZ)|C6VsB$;Z*DJkG+%dVYiV$0WNBe8aCz"
    "N5?N-}Jmj8JQjeg8Z-NvyA5aJp4EFGZ3OhR%3$;_^s)sdw#Xd+7<Nruo2ud~_jeWg6g-"
    "h1o2B!iQ5@9f!hAF!pWTeoiAy5Cj!zJD6eSkFsY!>V~!WOb7z-"
    "XF8~*@C56yjiiP%Ggz$d(*gPc~&rQ*<^XvWUTgPW#zqNRh^ZEx4pUF_kDk3V^)=OFN$W%W?8W)^0N7&tQs#ain56tXj*S<h-"
    "a5^eVJ!d`Ss6rS;)^7lb@DFmXs+=<0j^?@l%q=bqyGZHtG>?v4|UlE8o6`ulz-Gv&f1I`Fy{)QK%Hl`QpZlYp+<yM~k>f;Q{==NO"
    "|}Aea_;l=+9V8lVcy6Do$`t{RTcuxM#C0oxNj8Q&!netg`ze3F9i7!<2H_;YE2tbbrGxDpn&o?LKo{grx2-;;g#PY9{+7Wew1xI-"
    "6&CTxHFT(+$UnVAUAmD0`g((oHtYSo%X%E*F0|D)Y2wJf#OVq(#LRRhb}entZ1X5~xZ~lQ@T2@*aI$iYPDR)M$AC&)&s(meTqkR#"
    "jQ`y!tZU+S!d}Sq|)X#ow|hkEeid8qKPB&ICv?eKl)bXuvcfXpwl^^8m>-"
    "qnSoYndkAMX3;fZ4~Pgr^>RATh`th*8n<C}BL>~r_~Y>(j-!M9v;EV<v*^v?kvI1IczP4ZX%)lTKAINSut9vg-SGi5+uGV2?rhmF"
    "-W|R<J$`ipZ-@4a7st;IPfib`mxpKj_zZ`6_H3{@><<L&qvIEc(X&@?PtM@o<ALaZ_Udf^MRa=nkB9hlxD`F!dJ;X|-"
    "QjPay*fENJUNSA?w=eV9iE;=r_c9+(9ks4@Zg_6*x5?hUix&fJ51sCaP~NP8gD1*?%r^F_B7eqp6({e)2*jFk9W3r_qKPY$zbQ{&"
    "Xe6I$=2g(l5F`K;l`_@qvL1CfCXZBvi}k>@p0*E)_8S1U*xRr<9_xx&JYK)8IszAcKxKj@;6TRU%rM(p1%G5<?$)fDZ+NAh_258d"
    "3xEXOV1N8BIiZ8@rRRFe>~w-d%l1AoWR`DtXl|=JbyOa*&eVbgRLjK!#LjF8E$Rw#M9~SbSF)c$4?(`&-"
    "P}I)8THiGmYcDy`AB7dzem>-"
    "Q6_a+MaIh`8@({O%TAQ(<eKF?Wa$7c9Y#%JQ%R;^l7pU^c(In_IP`mY$wU^N&MvTliAKzoW_IQbSoLm_6F(B?vv@>P=H-"
    "Adl(=YY_WJYm<_jJoaD(<Sd(OL?+Gm7V7N8g-rIUQO@`?fgP~bGWGMh1OxbX6_sP!oknQZ)1GrTDc?P)KI6B-"
    "vd;132_r=?nC#RbKlIBAMd$ndoBQd!yt28RFSOvQ_LOE2*Hz1d%pkV;X1#VcD+Aq30fnJGZ$E=88tOckYCSPX=Zd!xbC@$nP*(S;"
    "vh>y#-"
    "s^y!cEKs;xFT#yChi7k&58nYpojreZc>4U+ivyfuvjmC;AXbYs0NJtx(Y6(mo}_PHR)B)($5mWhuwc+1?DV`re|v~O2Rk9`t2f6#"
    "9G?KxgTvP^Uj4)2LDVr2j5FwO?QFc+KRI~yGCDomKf{f+y|r<Aa{T%=NS0^M_uuS4I|IQ3&-"
    "VrcE*W0#|0xoyg3osb(Ewh3zkl+>i~Z;T20n|<0Osk7{WC7FfJFUy{2l~sT-"
    "1U8>El0t`uOqF$A9_s@n1iE{I^dZ|NYa)|M>LrKR<o^uTLNU8}}u!$K!3_;=QM^8BgB6c)_L7pFjxizj({HqmK=XxWHeNVwvaX{v"
    "F6%lIq^O?SrgNa8YkaO$2Y2MU%}LeF;Z=MSTBF3|cVtt}j{Pl`~KU85k%@IbX0QBUJ|k2Tfl<8)eD>?H9`i^mzp)Oi_c?!|j!gzg"
    "N~n0BYhVR*cW8B?}csN1&l<Jv6J!{0abfjtd&mi<b@H)u4Xr7@rS{z$AbO$eR@-Zl>T16p+T();2ZX8DmFPS(@Fh<-"
    "n_7B$vzLeLdn_JHZA$uix*VKS&fcsb^)iSZd<DhV21!F3MuFDz7(lcEylB*anyh=;1B#eN)220=R3LG#(Jk(^h6yr0j#HQCtBXk^"
    "EqT_0fztA{9+!DVJ5k_#8mt1D}kbcSG+|MkZr~#+Li8Vt)pEXvNDEC_xJsEvwvo`4C0Ol-ggWS)+)te*xCq1x{K)-JHc>s@-"
    "@quo~;%co*2o%TnYPu)L}o$<o9qBuWp3jwS>(jj~j&7En(tmVWI_29^QHdW9@T?`!=E%*6||c<8x)%sv2OktPC==)KVkY{2=ViIh"
    "atZyzELnw0y%ycfkBCaXv!j6z@uO``Z3cjyc{)U=(-"
    "f`9X3)!$pN$^!<|<e}!0H2f#XSu{_p_@?LS&*uiwu9Qg{VgG1)16$=<LV;-+ZD|^uz-k<{HhDAnVFIqvmSz_>BmDFrZQe-"
    "c2hp<3HF3_$#U@T(Z;o==fC#|{z#~93V3awVm(>mI>E;sNdU<(qfrhuuJav&RKuoOQ9pye`xX(1H9u|#?ejIZx2VPYfpb_VrV4Po"
    ";DXrrr%moZT;yAz^K4){F5_nfRNC)xu)vF`qvS~aOKBgfq1<Z)U9C<{3nhH2_#x{Vhy+)I=oJH11*3`|G7H@R5iDKIKh717GrEKP"
    "bgSg<b24ybTAvsX%ibVy;usR66&F|0-Mg-3}>FC8sJwz4EnjjP`Ucr1Saw~6e=>va%AC>6-_w@6bFZ81V)L$-"
    "ea|JX1R&~64M5x@7oE;6e(mS6Z(|3f*O_SkZ5QZA1ew?N-"
    "+yU=A82F(OrQik^ya`PC@K3J~!RR=M$l7>wh<3!gLuJo%4#EjNK$lRGkKS@66=)?!baPb}pE@>zk?=8f;AnzOTLRypr&%k9faDpp"
    ">#Nb5lK1I@^|%1f`a{4iH*7_eftUU2A}^<b|BWAd*^C1Ka}q&|WxAuE@Xi|bS7Yx!>ga_NBn>Rg<?3a>6#g*43iQpPe}&eY!*lra"
    "z`t8H0UV$<>|$7+N5n6L8GQiEYO?Hu`zR#15fB4AJRq<kBILNqtMGt+^>R-A`e=v*A^eK!>?dZC$bqXJ?NxBaH|i(2)-"
    "bP7PZmuykBe-E+Lnhr3|8)Wt4uI19GXgr)aP-"
    "1Nw!jrM8g3$L8@+wF+4ejGO)n>_5KiFT!05D{qidjsvD9}C37c3D*)C<MB@4aLoIP6WyBJ3bVHLIocjRAgWRY(@nH?|WW<7Kjlw("
    "6>p!}c!0vu5{5~ttrVB!g@B-2$`@la1Qg)a|cx0Qx>xmLF_5ZR2_9v>7W#r9K{{wvmB{zp&i~dsvWWxojC{`v7SVeM9K_HhU-"
    "L%9~{93?X5!|7vjebwtN%FsE#RVv1!`VPma={EdhFJH|Cu+(D%m`xpw4@P01t*utEJCE0)0$Q2-"
    "i+5^WR10CWQDzgBlpsZu+)QdQ}1a}+7lJHZK?$jJPp1Q&28lQ7KoWIm5+COA7XbsJ%?NQ=}yD4*pN?Ns;yMaK=a*wEa$SH;J*ZRH"
    "`&I*9>4|<7zCz(&rA5vv<P5LRAVLb$uQH8foXx1eG<##vFD38=g1<UMvSZi)jO1?FF19}3fOGyO#;=pXY;#y9&#O}zMHZ}4HTX~0"
    "9!Dh4N5*Hkh&`nf<HeucbabSc^o<OPf8^gZr#A~9k)n`5kY~XAQ52+EkP9H!lqg<m%*%H!pO_Bx|!U|J;);yKDAk#<;w~gh`I@jQ"
    "oekcRGJb^bz?KZXNigSLMd46L&6qd_MZ}IG1mf5?4jf=`Z4%0MT+V|0R(kJ(U>5-8+o_Q&4LBcKkP?jN=DJ0AKJVN7-"
    "q$inY%(yWnD?tS{I30$4rf><~94mnK>fOw{(QfArGnwU{^zUs%_`+neTn;`M>e}{-"
    "4XN2qdDm5oUD>z3;r?XA1L|B`MdqARpfjo3#Z!?WF4%vyRImb;Y`gE^u%lj1fUYZ3u%LLMFFy?-uG-"
    "o5|q3CEktT+kFC_BzIe=lW%HnVu~bCW89{<ciIN4*o=W>PJpdYuS>I;70WrpNJFq9iDHO<gd-"
    "9Pwx*B(Lz_3bDnNZhorG$zngAOFqS>Q%ekIg?5osCS)XJ9_&1?jST+2#eVJMvoP=Fj(*y@0?UG&Ln?*!S(qM!F$`5xA@J)XTp6Fv"
    "$qMqn-hr-"
    "*H*>2+Oa119AqBP2Ux$Tp+<)}AwT;#Y6D&%xqLw_CS5cQ>3H#qN1Qr05LfFW84B$Qo7|K>F}~j==)k5Nko=P`bPV6@OQ5hp~Z8*K"
    "6PC<;C9nqP(_7;>Pxp?EN`wV)VCw!ugGK1~OttOp1f9kdA;dL&sh#Z7G-"
    ">Xi8gCa)9b0nES!yC+h_4v$9Gt#>&+Qu8mr9Lj6_^^{eWx?Fh8Ivo0fF;`X`b(<=)TN7E6m7Iq^|gj@LuOm)}BU*B|>?*)e_;eNH"
    "vLpO$Gmysp^(R*^NA5lif4~b#gc_NTBH8`)p1}Oh9z=m2L7zv(;t+G0aWa~ZcK6XdAR<S+0nJd**=wyR<N9~`kaASn?A%baQ+Im&"
    "(tk&4YtV>n&w4Diy-_^xZ8M=8ZGdAW=bPE?&3%G7-"
    "6JqSprK1_A)R_O`4cc)N(yw*jrV3Z%bswiIKey&$=)(uZmc`EmO)xViIn^@W0POiZu5LI=Be`_My>_77U}7EK5^)6hjqwBcMM3?z"
    "M#lz)-ETll!!R)E8n-1XUf7@|h=8iRHW-+L)Im(&T@s(tCW3J8OGC-%HSz7mvH;kxy1QE><w+iZ@wjsUBi{&ZP$#?N2+O~-"
    "Hw9oQ?ubaP5s~~H;Evw~Bi!fD^hCUuvkEFF&03-"
    "g%T<bI4p#BV3+PFnt!=<7$wshhm7!Wx8Tw!sC=j!&EL}nqs*TVrvKG5jtrkWdQ&>~L9gB?uZsB9Jx@g8eSA4adtmv*Z3+QKv?7^C"
    "dk|9A_l?##YGH7HnB<|m@p%GO=k)aUJz`8LKl9<8HPzvyR1)e(jIszs2mGQQt@J-"
    "Ob5+?X$4WhJ!mqC>tO_Qgt0hd#TxrEiEk||*#mtrE3`2>@1k>kr2Hys~*CI8su0po_JEI0*{GGERMo*4-"
    "IEXA}3@sAl{<MvKRtz+Pyx}{v+d|5Xt+fTt|;DRje38<#c&zA*<#&43CAqrS}H9>^V`%Q^ao`4$4M(W`gVhL2`m`V(}RYrV^?=q~"
    "kNm=E$xkj)@lC!vw6$88^YJ@tbWn{nPkTJnQg&nHi`0r{DtAsWM46kM&=NhaBl}x~^c|2aw1w>_4g2il-"
    "9e|9Y_)5Ek0eWq)!3q$);MB=lEdas2Rp5%OE@J*XwDKJ^MFARzTr9Ye3>t^G?$c8{LFD_Cf)dU9d3nt+{0j8zs*=J`k!mamS(Mcr"
    "7VIY`E87C3#)zYA(o&2_6#!JYg&QVVRplkOu_$F;1>DL~ixDsqU9tWQ1VfzXLFNCLejEI5^kW}>hrfd@(v5Pc^1Ds>o)z%=-"
    "oX?QBr*&K+zCTX{^`f-AM116hu%FSSr!KkN6M9dqB0u^yk9T!40n&%bku}=Y^fo!5H8Bu2M{YDOn3!&9SE)@?0cT9BGi-"
    "7hSdxtW#j#oYDEkKYf+WyG7(9p?BWt++Hwk;_7Wy6M)TxoY8z76-mKV%1-Q{ECb3h-"
    "&h8VxB~>`8f_3HB(1ak%G@;G3EOQ%xY(tcy9Isr?&T%^0*<*@IPE{w0|8ON-0Kaxd4#7W|#~*@@wqeg3>R1B?hwOP-"
    "u~{s^G>mxJR4|k`Zf%k7X|8T>Pq&Q?t*Q&hDCux3VPOd1rNHe0f(xr#nObl8!U((A#k1bnCbXQwuU#S_sk>Toz;=MU3*F&r1cv>A"
    "45qd4uvSNk#mHb_`LRp#nvRB1cuZw$fwKaZS=}y49`QXKh^J=jcA4_1O@UVX74Yqs<}j-"
    "YoB3>LkG9sv>~B>*^G>b&h|7@#wMBKVCfkdV6%vRJw-^X?lv$4~mNFZ+E3bK5x$$fdX1dJsR10XbXhlmi1Ez)adiLn|2-"
    "b^*p@YS(7<A7PVL(b7gxBPu{lSYQ(a2ClQPp^2;~SHxm0_pQY8h+q(5w(P+B4D35*ETpyNs_uhx`RSDgb!V$V8E0>Z<wab)Vd;*f"
    "hCBcGZ21EB+H~IyoPMz9Mb=phfMR6mX|+mVgKT+d>5ZjiYB$MLuMXlDA@JEU`4XD<E=jg)=u(VX<@9qkB9UZc-qVm)>m3%y3>_kS"
    "3=XgwoS-Q5N)gyu#0SLjSz=19eZU?fchmMZCajlcH?;oEWD2SgMFfPPEuT+ML0?WTca2RW+VXpa-"
    "G^TYWc!8fq#UZQMt^f~Z`;Wc>=PI;!{qDv$kTGuzzrL(C6f0^{dyA{AiM(L{jV^&bBjraCsYRE7CZ*8@=V6-"
    "gpDk1YSS66NyWBECW1gV8Zi(Z|oVlB#ViL|&ZQ{{HmU$pKN5*0CGDqW`z1nq3QLF=N9U63F(*dhN)|edU0XH9C6=%atdU2n|Yx`1"
    "l;neLw76Gylip$HGSjAalGEp(bnatl_jwZ_o@yq0Fs*-q|u1<-c~;el@hoXm4=d^DO13$wk{prusV}-"
    "gZ94fBv${MO4!w3O?pE>8<Wdh(d4rE&2YQ_M%!Gdf?n-hsmgYxahna%3bTJsAUngaIPv6w7~!~DTVPwPK=}8(>R@TNFuMs6{iH-"
    "8bsCJVbDsSYJh!5_VHMZ@}9B9*q>!p-3Xc2aS8-oAU^2kdV@~X2aP<^vX8ydvD_J)uZpZ8NG!pX_%kplZoW*!i^Dg<%fzEO-"
    "m;FTsJH}l-"
    "<|@X%i=1zjHY=x_0P>2iRC!Q>@QV_bWG7yyp9f4H0wF~tS2QxPf3+t*BXTCn3#er&Q35LN0O%|&|n1bi&dLvN1Mk7NAeg2t9THLP"
    "Dy|ohX-`(nY-PI$Bz!^V+U070ZWuMQ51lVob?1vNACQ{Zm`p`*uF28U@ZFCFR#=wacJNi9Ji%JT*UbeJRSQuwdi_^agPhwKoOPfk"
    "AtnAw+*xeUszdj<JP%9ZI#=208Ogw1JxH=SoLQxaS9^=^Uni5h5ia=2{p+iwFSwzQQWe)JCD8)PU^%;N9h_hFb=I?g}{Pu5{d$d!"
    "H}=8By0>X$+6nq){|Ce9wcsdL5%>>k83I_!)T792mi8U)lCPzG_58a$hoalY9Jxw+~6GE4+vz>To2qC=q>T^H<u3m3e5c6v0SFSj"
    "2j&~I$wH$5fMI=qc0_8r6xk1Vz8-"
    "N+J2Uo$pqn^8{69jW~{=5$cf9uy;wdvhhPO^(AWSZvwDWR3W#d}3HumIq}VpjqqqsI*WPcvLFoO}vmbs3irZP8dqkv>Syu<Elz7Y"
    "vN{m<&GinHhU~||u;Es)?Y}m89YpE5zEdY%ppwnoTXu<p+`Q@vGiK@$@xIRCugPlTS?_oq%z{4UL4KFK<Ep~m0eu@Eff{oA7xu9P"
    "5LbsZYt!J(G=T>)o<n*9cE@APZp2rf&B_@)y89&(wIs&7ltB6Eu9QP1dpQ#*@2!as8p7DU-Se!|nkpR5AO@h3<?s=Ec2chzkrobA"
    "_-"
    "5bVau^W;MV8|*$esb*ItjI5zxRBpu4VQ7rN?!%V1OMa*Px$E6MkvGN>WCIl@`6>Ah6}#odEC`G3Fnjvn%y6UGd$cuNbv5VKIzeMu"
    "7gK;v~j=>w(_8YeMp!v=s1@9`%-"
    "4ys4#&a+0Jtv(71w)8y8KDbx)L0GExv+q(2hD4&}*TsdKF3FH0~Z5Mp;MyisC;w3&h}_=`|Ck2aEq#H@l*Hfo>LeW<la(QDk(iAZ"
    "l^S}~%#aRQ}Fp|jMah86`{+A37R!b-i0L}VRM<MB}pUX{t5LIsblZGi`T#XP2k-"
    "dj4mVK%li9oSVy2~Fv(BF64GDlTb`M3nBsdI96bDCaG1=cx7C5it-llk-sOSDJC?%v2(}F7TBg`GIXZh;2n$=8T8VC9C)O-"
    "U{pkA;@00I2>mHp}|A@K&Sw9=M<b0L04<pXJs+V@irdVKZ3o=Bw1Eed@c8%+QuF~7w_Bl86VleVKN*!RlBrj`dm@SM(Bg{&$d7|X"
    "{ACBVm<DqptELeQvL$ST6igJsTL3oVGfGa80UtrL%fdk;M$e~q`38YT_f*e?c>t1g@v-CbT5t{h9MUlT=uHdoiqPxdy}4R-"
    "qI)&Y<@l(Y|BH&5;h0JW1v)1#v=mUF$g%=kkeTlxwmsVTB>UoiTBb031VxCvtxfVK!R_?V=^84Scl~{;P3lrMw_n;9_qP!+ISk8I"
    "b-u2xhRz6JHk9o-qRHAyyXFmUbpc#{8tf?PUaUe@JBVnGaU4lwBU-"
    "{7oJrN_nuZ>wYme0ucL7b1I+O<n`VvJsWMxuG+<LLsX!L3_L`6sLd9^x48@^X$7ru6FT0de3<g-"
    "WFf@!xnFvm~FJJP4%O0EppOPwRS(PmzYBsb-x^7AfB)!M-172meoXv7Zy1a**FD>U0-"
    "FGs^r4RuYwE$w?vn9AXdQv@abJ$HRGE>IJ#X-"
    "80RKvN7(BFIch2cco7h34xzPiuCIV(1!TjByyzp5+MLPuB#>Ip*;QSWi)=!r!SZ5@PSg};~(Y*0VHHXrJd&e!(wX8wVD;|iDu@8b"
    ")~5YS~JxAgN{Z)hG7<C49ng%XesBrW*moZKcdJt2$)2@N~Wn3%9=>E!&N2ZOCYgCE2_84k8yT7t(*XHG=%*tCR=C+<V^ZAY7X&<5"
    "j~6Y#zRb+^9TyiXP$S|BmkigPepGOQOfsuqb64J1|#J*Jk1{>*h~oU@pWyr`XE@?7hsVdR;7EkTlv!#$XhmhQ&ATw=*(F4N|N@Xj"
    "vBLu6@5r2sIZ73L=Ytwr8@NNX;24W@c3))!-"
    "y7_ht{(#zVP65KvJ!D)+k5slzzinC=VUh0`isSsoAO4@Oi<trzT&1RSyykgP@5^Ai&>+0#K%E*nA81JIy63pdGz)=)Ea7Cqped4%"
    "WJAH2pC26z7*Ag>Z8j1Qy@y<6T2ssSousN=ShGmxy@%GMHXn~+)G||XB^qI}{tovQ@vhT*7WVv@71EL5zN!#5R*-"
    "c!zKzy5v<ZJEOQymB@1^c2LvGyXw>@5Gi<_+y^bbhmhb-Xt2W9n=}ETSLq+0}IWC@E|6I#OXU@)1@2+*mYsqT2F3k9k^Ho`)`KZD"
    "6YawdW(C@ylPw9rwO!dltj^=s1sug8`@xzFDuSLU8O);~b+uSchy`8|ICG!Vp#C;hY1T9zC&Aol};F*14^c$4yZdKe4I|I5eSgB0"
    "M%dv;?H+NFuUzLa|c-N;&{5N<xUUBQ+(gNCk6TH;2akP|{(v*b@-"
    "u7+TC5bgW=F8Ih?YF$QNWv@QiOqtw<IB24|olQYtB#U>xXo`k(M?HjSQ@>5w2v1Rd2E8s)Z)A*8Y`ni{>t=O4>jbrHY0YeTVE>`7"
    "ds6^MbGo|HLTh^nwLNK0XEs{cB-@w7E$Bg0d&KkO{TDm%hS>ud^@ubz#(%o#m_T0i%Oa}vf4y(0pPm{O2IJSh%T`-"
    "7bAf<jVr}#eTU=VNRooO&HN2lwP@FTHJ<ps?(KT~aDiq>d<Cv2H^To#ENpvNmNe^$PxF@!ZoEmwJRO7xrf^Ej!>Xf_1fptq{cY{+"
    "A|1FO+yJcY%jmTKZKV{ff1TunGG3_FD2TzePEd-"
    "J}?0^7L%u%B6ruB*^dMu`{(ohxOV$Y*Mh&1Bh>vzfZkPlW`EGSabH!_S`}NAh~|_TiAwxh>bV;DmD0yR)U9^su2F+~$Rcq?uLs^E"
    "CtAWawKF0Vf!+b8i&D)XRp;u0m&u8NSH*vEq=Of|HOGF1Z~>5F$C0(kV2|h6d|IT7`eeGmaUqco)a&x*6FpyJh_<0I((VDwEb*$A"
    "Z=vP_el0oCh~!qiXF{6P<=MY`go)iF>Pf$w(vkhAr;eCyt_SYY6VV8U4RhaDJWeR7}uWw$?6J^E(_FEIgB++vQ?DH@zrHMkDJY_("
    "gvX(jk~wCr`{8WTVy`Y}Ss7HeXB-V9Gxd(19{2I%v?X`W$s<s5oahUGgU61r^hhdx;@rA|h6r-"
    "Hh{iK27md|MdueOgIAPtIjOowFh`5d+FS$iZ+!ahxHz4mQ;x@C9-eS)YnP5-"
    "~i>(%A(Xq>!>b{PNbTB08Mq^S%%dW*IVZc<g}x1Mi7M|qS2(2T}&10D<u@AY>}5Y7{IU~uBB|d5Y()p(UH%LvPNIC1+5{px5d^it"
    "m#OoE96c1+SA$#Q;XuB5)@<b5}hKx<aZ(^LTOV1t_6~LHhlq$NDc&yO{!&sGaNS$_`SI*$M99wLC<SnWNX}JWUGLlx56h-"
    "_QVMw13O^n!{gSazr5UA*WU;eLhYKYsI!!LEKV-*=r7~pXJuK~Dt%2A&f$!^ax!wW)AajB9>P-ZjHMllyzS-iO#eepj^aecVKU4`"
    "%`6R@D}H@Q8-uT41}L;3F>LcL&1Ug*SG;fLFpnh)<vLjE&yk3HCTrLV8(N-"
    "rWr<pA)Xo*jbVVe>g8?nQyn2IwS((6~*L|*l4Og=$pp$U}c_$xPR1~2VV{E5fS|m7DH&ER7*nZyPMDJT;zL+NHbr2tgj&p?|{S+r"
    "t&z%BGYpr_PGQ#}5!DLdr*={x`S9@z1bA51wn22zAVgdOV+rV|#r|*Oc6wPx`=i(L?z|FvCVvKRVs-"
    "H1vS7#$8(1M^y(r$Zy{GbJih|)3{&6)Mu__m<fMDB<^D1=)P2+dI|BN0OOOSzARik;UW;%ySPXsTeI(7CIT^Q+1VbsK2*?$JcH#g"
    "qQ5hlqL|B3PJfd}yme+LhJn&PL0miMmE=S^sOUwYkkE?GeuP+TSbRIz$0-"
    "q1z0(&_d>C@;*%MrPCz$_#th~Y|rH93YI;${~ke`MR~YP*~vO?<`4wM+%FWp5`VsomS*K2Hw%nzJ-"
    "YLG?7zb^k(aF*`YVEBwkxNcjzU{-?%(HdEjr&;QQFVI(n|X-Am~Ng-Rbdwv*vTQFxd$`3gc^M`};f+)A~1_I&fTaLnrac9niGYsl"
    "-t?xa*mlPk%8C8*X@X|4ousU5Fhc#Az+D<n(Gg?ka`7&*<lu*RA|Z9?#-"
    "DS<;m|RA<0GARA<uhn8!p|B#2ekEQ1Ku;?4@=)xE(F4lMSq!duQubx*J*uwKgjk_uq>RP8pkr4(hNg>FH+bWN#ZQ&A*){be>DUc}"
    "jx<*qV_%fl~pK<&#LABaPlS8iZ)9C#-PP0}@9p<{Gdu%v)b{4hQk)qTN-ci;#p<;PqiIj7w=*eDeT?fW5ywbN^%__!zV(t|tq%PO"
    ";MmoqKoPo~@>2-$izR2CYKQB#h(}(iWf9q^9$I=cB%gcWRe^6t=-"
    "}hAud?}anaYhamdm7PE2VH)d4dUiE@iqXFFlnczGeaJNr$~PaqOO5kxY-a-@dYi@CO`Gc3v>T(Zh-"
    "&AQh*~8^_=B@?h=5N<r?8vR|1#^)@)Sdi8aS;wvVi>Ns!htwXd%kP^VSwzlU`b;W<G3Gj-keg!%(cXvuY*#d2@*!#@W|;R%+Sf9p"
    "$*(s7T9xCYKTXJg!%98g%AK;i*$)VKOpKKNkHg&q#|t$M32B2nP+J_+mQJ$t_Etec&SztJ<Tfx+gnCt|;%?y#0is*#l2QO(l6;?F"
    "q$XCb#HTT$wOV$W3b!%L#4jI6et;XT**r}y}kH}>BfaFigPMGGca+<I>81<{CB<-9-"
    "7@W?ytgtbwVM@B05Pyg}zFl}PFdtVIR!2w-5+X_1U&9{5IghQ^3C==qfsTlJDq7Tpg)<&kgAJ^Z>t1jp4-"
    "pa=ZV9m)Gs5lp(Se&@AeKjXg$@m+W<~>LcU9s7<hueG8x)pORBA=o7`8hV{&vfx-3!m1#n{**pb;ZP$vBo7w|DTxOp~G6-"
    "|36Sm0Rj{Q6aWAK2mk;8Apodx6~|Hl003D4000&M0000000000005+c00000Z)|C6VsB$;Z*DJNUukY>bYEXCaCuNm0Rj{Q6aWAK"
    "2mk;8App9$9E-"
    "sO007hr000^Q0000000000005+ch5!HnZ)|C6VsB$;Z*DJSVRT_%Y;R#?X>MmOaCuNm0Rj{Q6aWAK2mk;8AppQbc}4vT002BG000"
    "#L0000000000005+ce+2*lZ)|C6VsB$;Z*DJbWpr|BV{<NWc~DCM0u%!j000080000X0H*h+&UOO;05}N%02TlM00000000000Hl"
    "Gh5dZ*hY-wv^Z)0e0ZZB<bWMyn=Zf7oVc~DCM0u%!j000080000X0JaaH{wD<h0OJt=02lxO00000000000HlFg6#xKlY-wv^Z)0"
    "e0ZZC6nVr^-"
    "2b7^mGE^v8JO928D0~7!N00;m803iUb+xPL`3IG66B>(^y00000000000001_fwmd|0B>w*YhrI>Xm4&Wb~ImWVP|J-"
    "WiD`eP)h*<6ay3h000O8001EXlOA4Ya3cT!!hiq(8UO$Q0000000000q=CvM003`nX=`F{V`y(~FLpFvcWG;BaAjm^VJ>iaP)h{{"
    "000002LJ~E<^ccza!CLH000"
)

runtime_bytes = base64.b85decode(RUNTIME_ARCHIVE_B85.encode("ascii"))
if hashlib.sha256(runtime_bytes).hexdigest() != RUNTIME_ARCHIVE_SHA256:
    raise RuntimeError("Embedded Version 4-A runtime failed SHA-256 validation")
runtime_root = Path("/kaggle/working/olikbochon_v4a_runtime")
runtime_root.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(runtime_bytes)) as runtime_zip:
    names = runtime_zip.namelist()
    if any(Path(name).is_absolute() or ".." in Path(name).parts for name in names):
        raise RuntimeError("Unsafe embedded runtime path")
    runtime_zip.extractall(runtime_root)
sys.path.insert(0, str(runtime_root))

## Run the frozen reproduction

The official labels are evaluated before the authenticated test is loaded. Final outputs are a
validated candidate submission, honest grouped OOF probabilities, aligned test probabilities,
and an aggregate-only run summary. These are Kaggle runtime artifacts and must not be committed.

In [ ]:
from olikbochon.v4_kaggle import run_kaggle_v4a  # noqa: E402

RUN_SUMMARY = run_kaggle_v4a(Path("/kaggle/input"), Path("/kaggle/working"))

## Expected outputs

- `/kaggle/working/submission.csv` — validated candidate only; do not submit yet.
- `/kaggle/working/v4a_oof_probabilities.csv` — honest grouped OOF probabilities.
- `/kaggle/working/v4a_test_probabilities.csv` — ID-aligned label-1 probabilities.
- `/kaggle/working/v4a_run_summary.json` — aggregate metrics and configuration only.